# 🧑‍🏫 Regression Microprojects — Instructor Notebook
*Elements of Data Science* <br>
This notebook does three jobs, for **all of your classes at once**:

1. **Reads each class roster** exactly as exported and randomly assigns every student to one of five microprojects, dealt separately within each class
2. **Saves the assignments** to `/home/jovyan/shared-public/microprojects/assignments.csv`, where each student's `my_project(name)` finds them by hub login
3. Shows the live **gallery** of submitted figures and who has submitted

It is self-contained: the roster reading is done in this notebook, so you can see and edit it.

**Before you start:** upload one roster per class into **this folder on the hub**, named `class_<section number>.csv`, e.g. `class_01.csv` and `class_08.csv`. Files with other names (such as `class_list.csv` or `class_sample.csv`) are ignored. 🔒 Never commit rosters to GitHub: they hold student names and IDs.

## ⚙️ Setup

In [ ]:
import sys, os, glob, json, csv, re, hashlib
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))   # Fall 2026/shared-public
try:
    from init import *
except ModuleNotFoundError:
    print('init.py not found from this folder; continuing without the course styling.')

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Where students' notebooks look for assignments and put submissions — must match microproject_tools.py
MICRO_ROOT  = os.environ.get('MICRO_ROOT', '/home/jovyan/shared-public/microprojects')
SUBMIT_DIR  = os.path.join(MICRO_ROOT, 'submissions')
ASSIGN_FILE = os.path.join(MICRO_ROOT, 'assignments.csv')
os.makedirs(SUBMIT_DIR, exist_ok=True)
for folder in [MICRO_ROOT, SUBMIT_DIR]:
    try:
        os.chmod(folder, 0o777)          # every student must be able to write a submission
    except PermissionError:
        pass

projects = Table().with_columns(
    'project', make_array('MP1', 'MP2', 'MP3', 'MP4', 'MP5'),
    'title',   make_array('Carbon Dioxide at Mauna Loa', "Galton's Heights", 'Birth Weight and Gestation',
                          'Palmer Penguins', 'Periodic Trends'))

def user_key(login):
    "Scrambled stand-in for a hub login (same formula as microproject_tools.py), so IDs never go in the shared file."
    login = str(login).strip().lower()
    return hashlib.sha256(('eds-microprojects:' + login).encode()).hexdigest()[:12] if login else ''

print('Assignments file :', ASSIGN_FILE)
print('Submissions      :', SUBMIT_DIR)
projects

## 1. Read the Class Rosters
Each roster is read in the layout it was exported in. Two layouts are handled:

|Layout|First line|
|---|---|
|Registrar export, **no header**|`"Ahmed, Anas",tut13755,8,,,,`|
|Canvas export, **with header**|`Student,SIS Login ID,Section,,,,`|

Either way, the name is `"Last, First Middle"` and we keep the **first given name** and the **last name**. Canvas's *Test Student* (and any *Points Possible* row) is skipped. The class label comes from the file name: `class_01.csv` → `01`.

In [ ]:
def read_roster(path):
    "One class roster -> list of students, each a dict with section, first, last, login."
    with open(path, newline='', encoding='utf-8-sig') as f:
        rows = [[cell.strip() for cell in row] for row in csv.reader(f)]
    rows = [row for row in rows if any(row)]                     # drop blank lines

    # Header row? (Canvas: 'Student, SIS Login ID, Section')  Otherwise names are column 0, logins column 1.
    header = [cell.lower() for cell in rows[0]]
    name_col, login_col = 0, 1
    if 'student' in header or 'name' in header:
        name_col = header.index('student') if 'student' in header else header.index('name')
        for label in ['sis login id', 'login id', 'login', 'username', 'user']:
            if label in header:
                login_col = header.index(label)
                break
        rows = rows[1:]

    section = re.findall(r'\d+', os.path.basename(path))[-1].zfill(2)
    students = []
    for row in rows:
        full = row[name_col]
        if full.lower().startswith(('student, test', 'test student')) or 'points possible' in full.lower():
            continue                                             # Canvas test student / points row
        if ',' in full:
            last, given = full.split(',', 1)                     # 'Last, First Middle'
        else:
            given, last = full.rsplit(' ', 1) if ' ' in full else (full, '')
        given_names = given.split()
        if not given_names:
            continue
        login = row[login_col].split('@')[0].lower() if len(row) > login_col else ''
        students.append({'section': section, 'first': given_names[0], 'last': last.strip(), 'login': login})
    return students

class_files = sorted(glob.glob('class_[0-9]*.csv'))
print('Roster files found:', class_files)

all_students = []
for f in class_files:
    one_class = read_roster(f)
    print(f'  {f}: {len(one_class)} students, class {one_class[0]["section"]}')
    all_students = all_students + one_class

Now put everyone in one Table. Within each class, students who share a first name are shown with a last initial ("Justin C.", "Justin K.").

In [ ]:
sections = make_array(*[s['section'] for s in all_students])
firsts   = make_array(*[s['first'] for s in all_students])
lasts    = make_array(*[s['last'] for s in all_students])

display_names = make_array()
for i in np.arange(len(firsts)):
    same_class_same_first = np.count_nonzero((sections == sections.item(i)) & (firsts == firsts.item(i)))
    label = f'{firsts.item(i)} {lasts.item(i)[:1]}.' if same_class_same_first > 1 else firsts.item(i)
    display_names = np.append(display_names, label)

keys = make_array()
for s in all_students:
    # Use the login; if a roster has none, fall back to a key built from class + name
    keys = np.append(keys, user_key(s['login']) or user_key(f"{s['section']}|{s['first']}|{s['last']}"))

roster = Table().with_columns(
    'section',      sections,
    'display_name', display_names,
    'first_name',   firsts,
    'last_name',    lasts,
    'last_initial', make_array(*[l[:1].upper() for l in lasts]),
    'user_key',     keys)
roster.group('section')

👀 **Check the names** — the first few students of each class:

In [ ]:
for sec in np.unique(roster.column('section')):
    print('Class', sec)
    roster.where('section', sec).select('display_name', 'first_name', 'last_name').show(6)

## 2. Assign Projects
Within each class we shuffle the students with `np.random.choice(..., replace=False)` — a random **permutation** — and deal them to the projects like cards, so group sizes differ by at most one. The seed makes the deal reproducible.

**Saved assignments are respected.** Students who already have a project keep it. Students new to a roster are dealt into the smallest groups, and students no longer on a roster are dropped. Leftovers from earlier test runs (a class with no roster file, or names that aren't on any roster) are cleared automatically. To start a class over before students have seen it, list it in `redeal`.

In [ ]:
redeal = make_array()          # e.g. make_array('01') to re-deal class 01 from scratch
seed = 1013

# Previously saved assignments (if any) as a dict: user_key -> project
saved = {}
if os.path.exists(ASSIGN_FILE):
    old = Table.read_table(ASSIGN_FILE, dtype=str)
    if 'user_key' in old.labels and 'project' in old.labels:
        for key, pid in zip(old.column('user_key'), old.column('project')):
            saved[key] = pid
print(len(saved), 'saved assignments found')

def assign_class(class_roster, seed):
    "Keep saved projects; deal everyone else into the currently smallest groups."
    np.random.seed(seed)
    section = class_roster.column('section').item(0)
    use_saved = section not in redeal
    pids = projects.column('project')
    project_list = make_array()
    counts = {pid: 0 for pid in pids}
    for key in class_roster.column('user_key'):                       # first pass: who keeps a project
        pid = saved.get(key, '') if use_saved else ''
        project_list = np.append(project_list, pid)
        if pid:
            counts[pid] += 1
    kept = np.count_nonzero(project_list != '')
    new_students = np.random.choice(np.arange(class_roster.num_rows), class_roster.num_rows, replace=False)
    for i in new_students:                                             # second pass: deal the rest
        if project_list.item(i) == '':
            order = np.random.choice(pids, len(pids), replace=False)  # random tie-breaking
            smallest = min(order, key=lambda p: counts[p])
            project_list[i] = smallest
            counts[smallest] += 1
    print(f'Class {section}: {kept} kept their saved project, {class_roster.num_rows - kept} newly assigned')
    return class_roster.with_columns('project', project_list)

class_ids = np.unique(roster.column('section'))
assignments = assign_class(roster.where('section', class_ids.item(0)), seed)
for i in np.arange(1, len(class_ids)):
    assignments = assignments.append(assign_class(roster.where('section', class_ids.item(i)), seed + i))

assignments = assignments.select('section', 'display_name', 'first_name', 'last_initial', 'user_key', 'project')
assignments.to_csv(ASSIGN_FILE)
try:
    os.chmod(ASSIGN_FILE, 0o644)
except PermissionError:
    pass
print('Saved', assignments.num_rows, 'assignments to', ASSIGN_FILE)
assignments.pivot('project', 'section')

Each group, ready to show on screen. Set `show_class` to one class (e.g. `'08'`), or `None` for all.

In [ ]:
show_class = None

for sec in np.unique(assignments.column('section')):
    if show_class is not None and sec != show_class:
        continue
    print(f'── Class {sec} ──')
    one_class = assignments.where('section', sec)
    for pid, title in zip(projects.column('project'), projects.column('title')):
        names = sorted(one_class.where('project', pid).column('display_name'))
        print(f'  {pid} — {title:28s} ({len(names)})  {", ".join(names)}')
    print()

### 🎨 Board tools
Defines the glossy group board used by the draw, the group boards, and the drag-and-drop fixer below. It shows nothing — just run it.

In [ ]:
from IPython.display import display, HTML
import time

emoji = {'MP1': '🌍', 'MP2': '📏', 'MP3': '👶', 'MP4': '🐧', 'MP5': '⚛️'}
titles = dict(zip(projects.column('project'), projects.column('title')))

def board_html(revealed, message, highlight=None):
    "Five group Tables side by side, with a big message line on top."
    cards = ''
    for pid in projects.column('project'):
        names = revealed[pid]
        group = Table().with_columns(f'{len(names)} students', make_array(*names) if names else make_array())
        border = '#F2B705' if pid == highlight else '#A7A8AA'
        glow = 'box-shadow:0 0 18px #F2B705;' if pid == highlight else ''
        cards += (f'<div style="flex:1 1 180px;min-width:170px;border:3px solid {border};border-radius:12px;'
                  f'overflow:hidden;{glow}transition:all .3s">'
                  f'<div style="background:#A41E35;color:white;padding:8px 10px;font-weight:800;font-size:1.15em">'
                  f'{emoji[pid]} {pid}<div style="font-size:.75em;font-weight:600">{titles[pid]}</div></div>'
                  f'<div style="padding:4px 8px;font-size:1.05em">{group.as_html()}</div></div>')
    return (f'<div style="font-family:sans-serif">'
            f'<div style="font-size:2em;font-weight:900;margin:6px 0 12px;min-height:1.4em">{message}</div>'
            f'<div style="display:flex;flex-wrap:wrap;gap:10px;align-items:flex-start">{cards}</div></div>')

def draw(section, pause=0.8):
    "Reveal a class's saved groups one student at a time."
    one_class = assignments.where('section', section)
    n = one_class.num_rows
    revealed = {pid: [] for pid in projects.column('project')}
    board = display(HTML(board_html(revealed, f'🎲 Class {section}: {n} students, 5 projects…')), display_id=True)
    time.sleep(2 * pause)
    order = np.random.choice(np.arange(n), n, replace=False)          # random reveal order
    for k, i in enumerate(order):
        name, pid = one_class.column('display_name').item(i), one_class.column('project').item(i)
        board.update(HTML(board_html(revealed, f'🥁 Student {k + 1} of {n}…')))
        time.sleep(pause / 2)
        revealed[pid].append(name)
        board.update(HTML(board_html(revealed, f'<span style="color:#A41E35">{name}</span> → {emoji[pid]} {titles[pid]}', highlight=pid)))
        time.sleep(pause)
    board.update(HTML(board_html(revealed, f'🎉 Class {section}: all {n} students drawn!')))

### 📥 Submission reader
Defines how submissions are read (used by the group fixer and Part 3). It shows nothing — just run it.

In [ ]:
def load_submissions():
    "Read every submission record into a Table, reconnecting early ones to the roster."
    # Early submissions (handed in before the groups were saved) carry no login or class.
    # Reconnect them by the name the student typed: first name + last initial, which must be unique.
    by_name = {}
    for key, first, initial, sec in zip(assignments.column('user_key'), assignments.column('first_name'),
                                        assignments.column('last_initial'), assignments.column('section')):
        by_name.setdefault((first.lower(), initial.lower()), []).append((key, sec))
        by_name.setdefault((first.lower(), ''), []).append((key, sec))
    known = set(assignments.column('user_key'))

    records = []
    for p in sorted(glob.glob(os.path.join(SUBMIT_DIR, '*.json'))):
        try:
            r = json.load(open(p))
        except ValueError:
            print('⚠️ Could not read', os.path.basename(p)); continue
        r['png'] = p[:-5] + '.png'
        r['file'] = os.path.basename(p[:-5])
        key, sec = r.get('user_key', ''), str(r.get('section', '') or '')
        if key in known:
            r['matched'] = 'login'
            if not sec:
                sec = assignments.where('user_key', key).column('section').item(0)
        else:
            words = str(r.get('name', '')).replace('.', ' ').split()
            first = words[0].lower() if words else ''
            initial = words[-1][0].lower() if len(words) > 1 else ''
            hits = by_name.get((first, initial), [])
            if len(hits) == 1:
                key, sec = hits[0]
                r['matched'] = 'name'
            else:
                r['matched'] = 'unmatched' if not hits else 'ambiguous name'
        r['user_key'], r['section'] = key, sec
        if os.path.exists(r['png']):
            records.append(r)
        else:
            print('⚠️ Missing image for', r['file'])
    return Table().with_columns(
        'project',   make_array(*[r.get('project', '') for r in records]),
        'section',   make_array(*[r['section'] for r in records]),
        'name',      make_array(*[r.get('name', '') for r in records]),
        'user_key',  make_array(*[r['user_key'] for r in records]),
        'matched',   make_array(*[r['matched'] for r in records]),
        'r',         make_array(*[r.get('r', np.nan) for r in records]),
        'slope',     make_array(*[r.get('slope', np.nan) for r in records]),
        'intercept', make_array(*[r.get('intercept', np.nan) for r in records]),
        'caption',   make_array(*[r.get('caption', '') for r in records]),
        'submitted', make_array(*[r.get('submitted', '') for r in records]),
        'file',      make_array(*[r['file'] for r in records]),
        'png',       make_array(*[r['png'] for r in records]))

REFLECT_DIR = os.path.join(MICRO_ROOT, 'reflections')

def load_reflections():
    "Read every submitted reflection into a Table, linked to the roster like the figures."
    by_name = {}
    for key, first, initial, sec in zip(assignments.column('user_key'), assignments.column('first_name'),
                                        assignments.column('last_initial'), assignments.column('section')):
        by_name.setdefault((first.lower(), initial.lower()), []).append((key, sec))
        by_name.setdefault((first.lower(), ''), []).append((key, sec))
    known = set(assignments.column('user_key'))
    rows = []
    for p in sorted(glob.glob(os.path.join(REFLECT_DIR, '*.json'))):
        try:
            r = json.load(open(p))
        except ValueError:
            continue
        key, sec = r.get('user_key', ''), str(r.get('section', '') or '')
        if key in known:
            if not sec:
                sec = assignments.where('user_key', key).column('section').item(0)
        else:
            words = str(r.get('name', '')).replace('.', ' ').split()
            first = words[0].lower() if words else ''
            initial = words[-1][0].lower() if len(words) > 1 else ''
            hits = by_name.get((first, initial), [])
            if len(hits) == 1:
                key, sec = hits[0]
        project = r.get('project', '')
        if key in known and not project:
            project = assignments.where('user_key', key).column('project').item(0)
        rows.append((project, sec, r.get('name', ''), key, r.get('reflection', ''), r.get('submitted', '')))
    return Table().with_columns(
        'project',    make_array(*[x[0] for x in rows]),
        'section',    make_array(*[x[1] for x in rows]),
        'name',       make_array(*[x[2] for x in rows]),
        'user_key',   make_array(*[x[3] for x in rows]),
        'reflection', make_array(*[x[4] for x in rows]),
        'submitted',  make_array(*[x[5] for x in rows]))


### 🎲 The Draw — reveal one class's groups live
Project this cell on screen. Students are revealed **one at a time in random order** onto five group Tables, with a drumroll before each name. The projects were already assigned and saved above; the draw only reveals them, so running it never changes anyone's group.

Set `draw_class` to the class in front of you, e.g. `'03'`. Leave it as `None` to **skip the draw**, for example when you rerun the notebook for a class that has already seen its groups. `pause` is seconds per student (≈0.8 s × 58 students ≈ 1–2 minutes).

In [ ]:
draw_class = None          # e.g. '03' to run the live draw; None skips it
pause = 0.8                # seconds per student

if draw_class is None:
    print('Draw skipped (draw_class = None). The groups are shown in the next section.')
else:
    draw(draw_class, pause)

### 🗂️ The groups for every class
The same glossy board as the draw, shown instantly for each class. Set `board_class` to one class (e.g. `'03'`) or `None` for all.

In [ ]:
board_class = None

def show_board(section):
    "The finished group board for one class, from the saved assignments."
    one_class = assignments.where('section', section)
    revealed = {pid: sorted(one_class.where('project', pid).column('display_name'))
                for pid in projects.column('project')}
    display(HTML(board_html(revealed, f'Class {section}: {one_class.num_rows} students')))

for sec in np.unique(assignments.column('section')):
    if board_class is None or sec == board_class:
        show_board(sec)

### ✋ Fix a class's groups by drag and drop
For when students ended up in a different project than the draw gave them (for example, they picked their own before the groups were set).

1. Set `edit_class` and run the cell.
2. **Drag names** between the project columns. Moved names turn yellow.
3. A name tagged *submitted MP3* has already handed in a figure for a different project. **Match submitted projects** moves every such student to the project they actually did.
4. Click **💾 Save**. Nothing changes until you save. The board redraws, and students see their new project the next time they run `my_project()`.

**Undo all changes** puts every name back as it was before this run. Saved corrections are kept when you rerun the notebook.

In [ ]:
import ipywidgets as widgets
import uuid

edit_class = '01'          # which class to correct

def fix_groups(section):
    "Drag-and-drop board for one class. Changes are saved only when you click Save."
    global assignments
    one = assignments.where('section', section)
    if one.num_rows == 0:
        print(f'No class {section!r} in the assignments.')
        return
    # What each student actually submitted (if anything), to spot self-chosen projects
    # What each student actually submitted (if anything), to spot self-chosen projects.
    # load_submissions() also reconnects early submissions by name.
    subs = load_submissions()
    submitted = {k: p for k, p in zip(subs.column('user_key'), subs.column('project')) if k and p in titles}

    uid = uuid.uuid4().hex[:8]
    students = [{'key': k, 'name': n, 'project': p, 'submitted': submitted.get(k, '')}
                for k, n, p in zip(one.column('user_key'), one.column('display_name'), one.column('project'))]
    students.sort(key=lambda s: s['name'])

    columns = ''
    for pid in projects.column('project'):
        columns += (f'<div class="mp-col" data-project="{pid}">'
                    f'<div class="mp-head">{emoji[pid]} {pid} <span class="mp-n"></span>'
                    f'<div class="mp-title">{titles[pid]}</div></div><div class="mp-drop"></div></div>')

    board = f'''
<div id="mp-{uid}" class="mp-board">
<style>
  #mp-{uid} {{ font-family: sans-serif; }}
  #mp-{uid} .mp-cols {{ display: flex; gap: 10px; align-items: flex-start; flex-wrap: wrap; }}
  #mp-{uid} .mp-col {{ flex: 1 1 170px; min-width: 165px; border: 3px solid #A7A8AA; border-radius: 12px; overflow: hidden; background: #fff; }}
  #mp-{uid} .mp-col.over {{ border-color: #F2B705; box-shadow: 0 0 14px #F2B705; }}
  #mp-{uid} .mp-head {{ background: #A41E35; color: #fff; padding: 8px 10px; font-weight: 800; font-size: 1.1em; }}
  #mp-{uid} .mp-title {{ font-size: .75em; font-weight: 600; }}
  #mp-{uid} .mp-drop {{ min-height: 120px; padding: 5px; display: flex; flex-direction: column; gap: 3px; }}
  #mp-{uid} .mp-card {{ padding: 3px 8px; line-height: 1.3; border: 1px solid #ccc; border-radius: 7px; background: #FBF9F6; cursor: grab; font-size: 1.02em; user-select: none; }}
  #mp-{uid} .mp-card.moved {{ background: #FFF4CC; border-color: #E0B000; font-weight: 700; }}
  #mp-{uid} .mp-card .mp-flag {{ color: #A41E35; font-size: .8em; }}
  #mp-{uid} .mp-bar {{ display: flex; gap: 8px; margin: 0 0 10px; align-items: center; flex-wrap: wrap; }}
  #mp-{uid} button {{ padding: 6px 12px; border-radius: 8px; border: 1px solid #A41E35; background: #fff; color: #A41E35; font-weight: 700; cursor: pointer; }}
  #mp-{uid} .mp-status {{ color: #555; }}
</style>
<div class="mp-bar">
  <b style="font-size:1.25em">Class {section}: drag names between projects</b>
  <button class="mp-match">Match submitted projects</button>
  <button class="mp-reset">Undo all changes</button>
  <span class="mp-status"></span>
</div>
<div class="mp-cols">{columns}</div>
</div>
<script>
(function () {{
  const root = document.getElementById('mp-{uid}');
  const students = {json.dumps(students)};
  let dragged = null;
  function card(s) {{
    const c = document.createElement('div');
    c.className = 'mp-card'; c.draggable = true; c.dataset.key = s.key; c.dataset.orig = s.project;
    c.dataset.submitted = s.submitted;
    c.innerHTML = s.name.replace(/[&<>]/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;'}})[ch]) +
      (s.submitted && s.submitted !== s.project ? ' <span class="mp-flag">submitted ' + s.submitted + '</span>' : '');
    c.addEventListener('dragstart', e => {{ dragged = c; e.dataTransfer.setData('text/plain', s.key); }});
    return c;
  }}
  function place(projectOf) {{
    root.querySelectorAll('.mp-drop').forEach(d => d.innerHTML = '');
    students.forEach(s => root.querySelector('.mp-col[data-project="' + projectOf(s) + '"] .mp-drop').appendChild(card(s)));
    sync();
  }}
  function sync() {{
    const result = {{}}; let moved = 0;
    root.querySelectorAll('.mp-col').forEach(col => {{
      const cards = col.querySelectorAll('.mp-card');
      col.querySelector('.mp-n').textContent = '(' + cards.length + ')';
      cards.forEach(c => {{
        result[c.dataset.key] = col.dataset.project;
        const m = col.dataset.project !== c.dataset.orig; c.classList.toggle('moved', m); if (m) moved++;
        const f = c.querySelector('.mp-flag');
        if (f) f.style.display = (c.dataset.submitted === col.dataset.project) ? 'none' : '';
      }});
    }});
    root.querySelector('.mp-status').textContent = moved + ' change' + (moved === 1 ? '' : 's') + ' not yet saved';
    const box = document.querySelector('.mpsync-{uid} input, .mpsync-{uid} textarea');
    if (box) {{
      box.value = JSON.stringify(result);
      box.dispatchEvent(new Event('input', {{ bubbles: true }}));
      box.dispatchEvent(new Event('change', {{ bubbles: true }}));
    }}
  }}
  root.querySelectorAll('.mp-col').forEach(col => {{
    col.addEventListener('dragover', e => {{ e.preventDefault(); col.classList.add('over'); }});
    col.addEventListener('dragleave', () => col.classList.remove('over'));
    col.addEventListener('drop', e => {{
      e.preventDefault(); col.classList.remove('over');
      if (dragged) {{ col.querySelector('.mp-drop').appendChild(dragged); dragged = null; sync(); }}
    }});
  }});
  root.querySelector('.mp-reset').onclick = () => place(s => s.project);
  root.querySelector('.mp-match').onclick = () => place(s => s.submitted || s.project);
  place(s => s.project);
}})();
</script>'''

    current = {s['key']: s['project'] for s in students}
    pending = dict(current)
    sync_box = widgets.Text(value=json.dumps(current), layout=widgets.Layout(display='none'))
    sync_box.add_class(f'mpsync-{uid}')
    save = widgets.Button(description=f'💾 Save class {section}', button_style='danger',
                          layout=widgets.Layout(width='220px', height='40px'))
    out = widgets.Output()

    def on_change(change):
        try:
            pending.update(json.loads(change['new']))
        except ValueError:
            pass

    def on_save(_):
        global assignments
        new_projects = make_array(*[pending.get(k, p) if s == section else p for k, p, s in
                                    zip(assignments.column('user_key'), assignments.column('project'),
                                        assignments.column('section'))])
        changed = np.count_nonzero(new_projects != assignments.column('project'))
        assignments = assignments.with_column('project', new_projects)
        assignments.to_csv(ASSIGN_FILE)
        with out:
            out.clear_output()
            print(f'Saved {changed} change(s) for class {section} to {ASSIGN_FILE}. '
                  'Students see their new project the next time they run my_project().')
            show_board(section)

    sync_box.observe(on_change, names='value')
    save.on_click(on_save)
    display(HTML(board))
    display(widgets.VBox([sync_box, save, out]))

fix_groups(edit_class)

## 3. Submissions 🖼️
Rerun these cells any time during class. Each submission is a PNG plus a small JSON file with the student's *r*, slope, intercept, caption, and class.

In [ ]:
submissions = load_submissions()
print(submissions.num_rows, 'submissions found in', SUBMIT_DIR)
if submissions.num_rows:
    print(submissions.group(['section', 'matched']).relabeled('section', 'class').relabeled('matched', 'linked by'))
    problems = submissions.where('matched', are.contained_in(make_array('unmatched', 'ambiguous name')))
    if problems.num_rows:
        print('⚠️ Not linked to a student on any roster. They appear in the gallery with show_class = None; '
              'ask these students to rerun submit_plot, or use the ➕ cell below:')
        problems.select('file', 'name', 'project', 'matched').show()
    submissions.drop('png', 'user_key').sort('submitted', descending=True).show(10)

### ➕ Add a missing submission by hand
The easiest fix is for the student to **rerun the `submit_plot(...)` cell** in their notebook: a new submission replaces the old one and links to their roster entry automatically.

If they can't, have them download their figure (right-click → *Save image*, or the PNG saved beside their notebook) and send it to you. Upload it into this folder, then fill in and run the cell below. Use their name exactly as it appears in the groups, e.g. `'Chloe B.'`. `r` and `slope` are optional.

In [ ]:
import shutil, datetime

def add_submission(png_file, name, section, project, caption='', r=np.nan, slope=np.nan, intercept=np.nan):
    "Put a student's figure into the gallery for them, linked to their roster entry."
    match = assignments.where('section', str(section)).where('display_name', name)
    if match.num_rows != 1:
        print(f'No single student {name!r} in class {section}. Names in that class:')
        print(', '.join(sorted(assignments.where('section', str(section)).column('display_name'))))
        return
    key = match.column('user_key').item(0)
    first, initial = match.column('first_name').item(0), match.column('last_initial').item(0)
    stem = f'{project}_{section}_{first}{initial}_{key[:4]}'
    shutil.copy(png_file, os.path.join(SUBMIT_DIR, stem + '.png'))
    record = {'name': f'{first} {initial}.', 'first_name': first, 'project': project, 'user_key': key,
              'section': str(section), 'r': float(r), 'slope': float(slope), 'intercept': float(intercept),
              'caption': caption, 'submitted': datetime.datetime.now().isoformat(timespec='seconds'),
              'added_by_instructor': True}
    with open(os.path.join(SUBMIT_DIR, stem + '.json'), 'w') as f:
        json.dump(record, f, indent=1)
    print(f'✅ Added {stem}.png to the gallery. Rerun the cells below to see it.')

# Example — remove the # and fill in:
# add_submission('chloe_figure.png', name='Chloe B.', section='01', project='MP5',
#                caption='One sentence describing the finding.', r=0.90, slope=343)

### Who has submitted?

In [ ]:
show_class = None          # e.g. '08'

done = make_array(*['✅' if key in submissions.column('user_key') else '—'
                    for key in assignments.column('user_key')])
status = assignments.with_columns('submitted', done)
if show_class is not None:
    status = status.where('section', show_class)
print(np.count_nonzero(status.column('submitted') == '✅'), 'of', status.num_rows, 'students have submitted')
status.pivot('submitted', 'section').show()
status.select('section', 'project', 'display_name', 'submitted').sort('display_name').sort('section').show()

### Show the figures
Each project gets its own banner; each figure is a card labeled **first name, last initial, and section** (e.g. *Daniel H. · 08*), with the caption and the student's numbers. Set `show_class` above to show one class, and `card_width` to make cards larger or smaller.

The same gallery is also **saved as a web page**, `microproject_gallery_08.html` (or `_all`), in this folder. The figures are embedded, so it opens in any browser outside Jupyter, even offline: download it, email it, or post it on Canvas.

In [ ]:
import base64, html
from IPython.display import display, HTML

card_width = 460           # pixels; cards wrap to fit the screen
emoji = {'MP1': '🌍', 'MP2': '📏', 'MP3': '👶', 'MP4': '🐧', 'MP5': '⚛️', 'DEMO': '🚗'}
titles = dict(zip(projects.column('project'), projects.column('title')))
titles['DEMO'] = 'Instructor demo'

# Name labels come from the roster (matched by login): 'First L. · section', whatever the student typed
roster_label = {}
for key, first, initial, sec in zip(assignments.column('user_key'), assignments.column('first_name'),
                                    assignments.column('last_initial'), assignments.column('section')):
    roster_label[key] = f'{first} {initial}. · {sec}' if initial else f'{first} · {sec}'

def student_label(key, typed_name, section):
    "Roster label for this login; otherwise built from the name the student typed."
    if key in roster_label:
        return roster_label[key]
    parts = str(typed_name).split()
    first = parts[0] if parts else '?'
    initial = f' {parts[-1][0].upper()}.' if len(parts) > 1 else ''
    return f'{first}{initial}' + (f' · {section}' if section else '')

def card(i, sub):
    "One figure card: name label, image, caption, numbers."
    with open(sub.column('png').item(i), 'rb') as f:
        img = base64.b64encode(f.read()).decode()
    label = html.escape(student_label(sub.column('user_key').item(i), sub.column('name').item(i),
                                      sub.column('section').item(i)))
    caption = html.escape(sub.column('caption').item(i)) or '<i>(no caption)</i>'
    numbers = f"r = {sub.column('r').item(i):.3f} &nbsp;·&nbsp; slope = {sub.column('slope').item(i):.4g}"
    return (f'<div style="background:white;border:1px solid #ccc;border-radius:10px;padding:10px 12px;'
            f'box-shadow:0 2px 6px rgba(0,0,0,.15)">'
            f'<div style="font-size:1.3em;font-weight:800;margin-bottom:4px">{label}</div>'
            f'<img src="data:image/png;base64,{img}" alt="Figure by {label}" style="width:100%;border-radius:6px">'
            f'<div style="font-size:1.05em;line-height:1.35;margin-top:8px">{caption}</div>'
            f'<div style="font-size:.85em;color:#555;margin-top:6px">{numbers}</div></div>')

def gallery_html(pid, section=None):
    "A banner for one project, then every submitted figure for it as cards."
    sub = submissions.where('project', pid)
    if section is not None:
        sub = sub.where('section', section)
    order = np.argsort([student_label(k, n, s) for k, n, s in
                        zip(sub.column('user_key'), sub.column('name'), sub.column('section'))])
    banner = (f'<div style="background:#A41E35;color:white;border-radius:10px;padding:12px 18px;margin:28px 0 12px;'
              f'font-size:1.6em;font-weight:900">{emoji.get(pid, "")} {pid}: {html.escape(titles.get(pid, pid))}'
              f'<span style="font-size:.6em;font-weight:600"> &nbsp;— {sub.num_rows} figure{"s" if sub.num_rows != 1 else ""}'
              f'{" · class " + section if section else ""}</span></div>')
    if sub.num_rows == 0:
        return banner + '<div style="color:#777;margin-left:6px">No submissions yet.</div>'
    cards = ''.join(card(i, sub) for i in order)
    return banner + (f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax({card_width}px,1fr));'
                     f'gap:16px;align-items:start">{cards}</div>')

sections_html = [gallery_html(pid, section=show_class) for pid in projects.column('project')]
for part in sections_html:
    display(HTML(part))

# Save the same gallery as a stand-alone web page (Part 2, reflections, is added by the next cell)
class_label = 'All classes' if show_class is None else f'Class {show_class}'
gallery_file = f'microproject_gallery_{show_class or "all"}.html'
n_figures = submissions.num_rows if show_class is None else submissions.where('section', show_class).num_rows

def save_gallery_page(extra_parts=()):
    "Write the gallery (figures, then any extra parts) to one stand-alone web page."
    page = ('<!DOCTYPE html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">'
            '<title>Microproject Results Gallery</title></head>'
            '<body style="font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#FBF9F6;color:#221C1E;'
            'max-width:1500px;margin:0 auto;padding:24px 20px 48px">'
            '<div style="background:#221C1E;color:#FBF9F6;border-radius:12px;padding:22px 26px">'
            '<div style="font-size:.9em;font-weight:700;letter-spacing:2px;text-transform:uppercase;color:#F2B8C4">'
            'Elements of Data Science · Regression Microprojects</div>'
            '<div style="font-size:2.2em;font-weight:900">Results Gallery</div>'
            f'<div style="color:#D9CFD2">{class_label} · {n_figures} figures</div></div>'
            '<h2 style="margin:30px 0 0">Part 1 · Figures</h2>'
            + ''.join(sections_html) + ''.join(extra_parts) + '</body></html>')
    with open(gallery_file, 'w', encoding='utf-8') as f:
        f.write(page)

save_gallery_page()
print(f'Saved {gallery_file} — open it in any browser.')
display(HTML(f'<a href="{gallery_file}" target="_blank" download style="font-size:1.2em;font-weight:700">⬇ Download the gallery page</a>'))

### 💬 Part 2 · Reflections
For each project: a **word cloud** of the words students used most in their reflections (common words like *the* and *was* are left out; bigger = used more often; hover a word for its count), then every student's reflection. Uses `show_class` from above.

Students send their reflection with `submit_reflection(name)` at the end of their notebook, after saving it. A student whose notebook is older than that cell can save their notebook and paste this into a new cell:
```python
import importlib, microproject_tools; importlib.reload(microproject_tools); microproject_tools.submit_reflection(name)
```
Set `reflections_in_page = False` to keep reflections out of the saved gallery page (they still show here).

In [ ]:
import re
from collections import Counter

reflections_in_page = True
cloud_words = 45           # how many words in each cloud

stop_words = set("""
a about above after again against all also am an and any are as at be because been before being below between both
but by can could did do does doing done down during each even every few for from further get got had has have having
he her here hers him his how i if in into is it its itself just like lot lots made make many me more most much my
myself no nor not now of off on once only or other our out over own really same she should so some something such
than that the their them then there these they thing things this those through to too under until up us very was we
were what when where which while who whom why will with would you your yours i'm it's didn't don't wasn't couldn't
think felt feel bit kind way well one two first overall able still since though
""".split())

def word_cloud_html(texts):
    "A word cloud of the most common meaningful words, as HTML (no extra packages needed)."
    words = []
    for t in texts:
        words += [w.strip("'") for w in re.findall(r"[a-zA-Z][a-zA-Z'-]+", str(t).lower())]
    counts = Counter(w for w in words if len(w) > 2 and w not in stop_words)
    top = counts.most_common(cloud_words)
    if not top:
        return '<div style="color:#777">No words yet.</div>'
    biggest = top[0][1]
    palette = ['#A41E35', '#221C1E', '#7A1626', '#5B4A50', '#8A5F00', '#2F5D8A']
    np.random.seed(7)
    order = np.random.choice(np.arange(len(top)), len(top), replace=False)     # mix big and small words
    spans = ''
    for i in order:
        word, n = top[i]
        size = 15 + 47 * np.sqrt(n / biggest)
        weight = 800 if n / biggest > 0.5 else 600
        spans += (f'<span title="{n} use{"s" if n != 1 else ""}" style="font-size:{size:.0f}px;font-weight:{weight};'
                  f'color:{palette[i % len(palette)]};margin:2px 10px;line-height:1.1;display:inline-block">'
                  f'{html.escape(word)}</span>')
    return ('<div style="background:white;border:1px solid #ddd;border-radius:12px;padding:18px 16px;'
            f'text-align:center;margin-bottom:14px">{spans}</div>')

def reflection_card(label, text):
    "One student's reflection as a card."
    body = html.escape(str(text)).replace('\n\n', '<br><br>').replace('\n', '<br>')
    return ('<div style="background:white;border:1px solid #ccc;border-left:5px solid #A41E35;border-radius:10px;'
            'padding:12px 16px;box-shadow:0 2px 6px rgba(0,0,0,.08)">'
            f'<div style="font-size:1.15em;font-weight:800;margin-bottom:6px">{html.escape(label)}</div>'
            f'<div style="font-size:1.02em;line-height:1.45">{body}</div></div>')

reflections = load_reflections()
if show_class is not None:
    reflections = reflections.where('section', show_class)
print(reflections.num_rows, 'reflections' + (f' for class {show_class}' if show_class else ''))

part2 = ['<h2 style="margin:40px 0 0">Part 2 · Reflections</h2>']
for pid in projects.column('project'):
    sub = reflections.where('project', pid)
    banner = ('<div style="background:#221C1E;color:white;border-radius:10px;padding:12px 18px;margin:28px 0 12px;'
              f'font-size:1.5em;font-weight:900">💬 {emoji.get(pid, "")} {pid}: {html.escape(titles.get(pid, pid))}'
              f'<span style="font-size:.6em;font-weight:600"> &nbsp;— {sub.num_rows} '
              f'reflection{"s" if sub.num_rows != 1 else ""}</span></div>')
    if sub.num_rows == 0:
        part2.append(banner + '<div style="color:#777;margin-left:6px">No reflections yet.</div>')
        continue
    labels = [student_label(k, n, s) for k, n, s in
              zip(sub.column('user_key'), sub.column('name'), sub.column('section'))]
    cards = ''.join(reflection_card(labels[i], sub.column('reflection').item(i)) for i in np.argsort(labels))
    part2.append(banner + word_cloud_html(sub.column('reflection')) +
                 '<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(420px,1fr));gap:14px">'
                 f'{cards}</div>')

for part in part2:
    display(HTML(part))
if reflections_in_page:
    save_gallery_page(part2)
    print(f'Added reflections to {gallery_file}.')

## 🎬 Results Slide Show
Builds a stand-alone slide show of the submitted figures: a title slide, then for each project a **project slide** (question, twist, number of figures, the class's average *r*) followed by **one slide per student figure** with their name and caption.

Set `slides_class` to one class (e.g. `'08'`) or `None` for all, then run the cell. It writes `microproject_results_08.html` (or `_all`) into this folder; images are embedded, so the file works anywhere, even offline.

**To present:** in the file browser, right-click the file → **Open in New Browser Tab** (or open it and click **Trust HTML**). Then use ← → or click to move, and press **F** for full screen. Rerun the cell after late submissions to rebuild.

In [ ]:
import base64, html
from IPython.display import display, HTML

slides_class = None          # e.g. '08' for one class, or None for every class
shuffle_order = False        # True = students in random order within each project

project_info = {
    'MP1': ('🌍', 'Carbon Dioxide at Mauna Loa', 'How fast is CO₂ rising — and is it speeding up?',
            'r = 0.99, yet the errors curve: the rise is now 2.3× faster than in the 1960s.'),
    'MP2': ('📏', "Galton's Heights", 'Do tall parents have tall children?',
            'Where “regression” got its name: children of extreme parents are closer to average.'),
    'MP3': ('👶', 'Birth Weight and Gestation', 'How much does each extra day of pregnancy add?',
            'At the same gestation, babies of smokers are lighter. Cause, or association?'),
    'MP4': ('🐧', 'Palmer Penguins', 'Can flipper length predict body mass?',
            'r = 0.87 for all penguins, but much weaker within each species.'),
    'MP5': ('⚛️', 'Periodic Trends', 'Does electronegativity track ionization energy?',
            'A strong r, but both share one cause: the pull of the nucleus.'),
}
# Name labels ('First L. · section') come from student_label(), defined in the gallery cell above

def esc(text):
    return html.escape(str(text))

def image_data(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

chosen = submissions if slides_class is None else submissions.where('section', slides_class)
class_label = 'All classes' if slides_class is None else f'Class {slides_class}'

slides = [f'''<section class="title">
  <p class="eyebrow">Elements of Data Science · Regression Microprojects</p>
  <h1>Results Gallery</h1>
  <p class="sub">{esc(class_label)} · {chosen.num_rows} figures · 5 projects</p>
</section>''']

for pid, (icon, title, question, twist) in project_info.items():
    sub = chosen.where('project', pid)
    if sub.num_rows == 0:
        continue
    rs = sub.column('r')
    order = np.random.choice(np.arange(sub.num_rows), sub.num_rows, replace=False) if shuffle_order \
            else np.argsort([student_label(k, n, s) for k, n, s in zip(sub.column('user_key'), sub.column('name'), sub.column('section'))])
    slides.append(f'''<section class="project">
  <div class="icon">{icon}</div>
  <p class="eyebrow">{pid}</p>
  <h1>{esc(title)}</h1>
  <p class="question">{esc(question)}</p>
  <p class="twist"><b>Twist:</b> {esc(twist)}</p>
  <p class="stats">{sub.num_rows} figure{'s' if sub.num_rows != 1 else ''} · average r = {np.mean(rs):.3f}
     · range {min(rs):.3f} to {max(rs):.3f}</p>
</section>''')
    for i in order:
        who = student_label(sub.column('user_key').item(i), sub.column('name').item(i), sub.column('section').item(i))
        caption = sub.column('caption').item(i) or '(no caption)'
        slides.append(f'''<section class="figure">
  <div class="bar"><span>{icon} {pid} · {esc(title)}</span><span>{esc(who)}</span></div>
  <img src="data:image/png;base64,{image_data(sub.column('png').item(i))}" alt="Figure by {esc(who)}">
  <p class="caption">{esc(caption)}</p>
  <p class="numbers">r = {sub.column('r').item(i):.3f} · slope = {sub.column('slope').item(i):.4g}
     · intercept = {sub.column('intercept').item(i):.4g}</p>
</section>''')

style = '''
* { box-sizing: border-box; margin: 0; }
html, body { height: 100%; background: #1d1719; font-family: -apple-system, "Segoe UI", Roboto, Arial, sans-serif; }
#deck { position: absolute; width: 1920px; height: 1080px; left: 50%; top: 50%; transform-origin: center; }
section { position: absolute; inset: 0; display: none; flex-direction: column; padding: 70px 110px; background: #FBF9F6; color: #221C1E; }
section.on { display: flex; }
.title, .project { justify-content: center; background: #221C1E; color: #FBF9F6; }
.project { background: linear-gradient(135deg, #7A1626, #A41E35); }
.eyebrow { font-size: 30px; font-weight: 700; letter-spacing: 4px; text-transform: uppercase; color: #F2B8C4; }
.title h1 { font-size: 150px; font-weight: 900; margin: 16px 0; }
.project h1 { font-size: 110px; font-weight: 900; line-height: 1.05; margin: 10px 0 34px; }
.sub { font-size: 44px; color: #D9CFD2; }
.icon { font-size: 120px; }
.question { font-size: 54px; font-weight: 700; line-height: 1.25; max-width: 1500px; }
.twist { font-size: 40px; line-height: 1.35; margin-top: 26px; max-width: 1550px; color: #F6E3E7; }
.stats { font-size: 32px; margin-top: 46px; color: #F2B8C4; }
.bar { display: flex; justify-content: space-between; font-size: 34px; font-weight: 800; color: #A41E35;
       border-bottom: 4px solid #A41E35; padding-bottom: 14px; margin-bottom: 22px; }
.figure img { flex: 1; min-height: 0; width: 100%; object-fit: contain; }
.caption { font-size: 40px; line-height: 1.3; margin-top: 22px; font-weight: 600; }
.numbers { font-size: 28px; color: #6a5f63; margin-top: 10px; }
#count { position: fixed; right: 18px; bottom: 12px; color: #a99; font-size: 16px; }
'''
script = '''
const s = [...document.querySelectorAll('section')]; let k = 0;
function show(i) { k = Math.max(0, Math.min(s.length - 1, i)); s.forEach((x, j) => x.classList.toggle('on', j === k));
  document.getElementById('count').textContent = (k + 1) + ' / ' + s.length + '   ← → to move · F full screen'; }
function fit() { const d = document.getElementById('deck'); const z = Math.min(innerWidth / 1920, innerHeight / 1080);
  d.style.transform = 'translate(-50%, -50%) scale(' + z + ')'; }
addEventListener('keydown', e => {
  if (['ArrowRight', 'ArrowDown', ' ', 'PageDown'].includes(e.key)) show(k + 1);
  if (['ArrowLeft', 'ArrowUp', 'PageUp'].includes(e.key)) show(k - 1);
  if (e.key === 'Home') show(0); if (e.key === 'End') show(s.length - 1);
  if (e.key === 'f' || e.key === 'F') document.fullscreenElement ? document.exitFullscreen() : document.documentElement.requestFullscreen(); });
addEventListener('click', e => show(e.clientX < innerWidth / 3 ? k - 1 : k + 1));
addEventListener('resize', fit); fit(); show(0);
'''
page = ('<!DOCTYPE html><html><head><meta charset="utf-8"><title>Microproject Results</title>'
        f'<style>{style}</style></head><body><div id="deck">{"".join(slides)}</div>'
        f'<div id="count"></div><script>{script}</script></body></html>')

slideshow_file = f'microproject_results_{slides_class or "all"}.html'
with open(slideshow_file, 'w', encoding='utf-8') as f:
    f.write(page)
print(f'Wrote {slideshow_file}: {len(slides)} slides ({chosen.num_rows} figures)')
display(HTML(f'<a href="{slideshow_file}" target="_blank" style="font-size:1.3em;font-weight:700">▶ Open the slide show</a>'))

## 4. Discussion: Did Everyone Get the Same Line?
Students on the same project used the same data, so their full-data *r* and slope **should agree**. Disagreement is a good discussion starter: a toolbox bug, a subset of the data (the twist line), or swapped $x$ and $y$ ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)).

In [ ]:
if submissions.num_rows:
    submissions.select('project', 'r', 'slope').group('project', np.mean).show()

**Gallery-walk prompts**
- Which figure would you put in a newspaper? Why?
- Which project shows that a large *r* does not guarantee a straight line is the right model?
- In which projects did splitting the data into groups change the story?
- Which findings could you state as **causes**, and which only as **associations**?